# Права и эксплуатация HDFS — 30 заданий

Практика выполняется на eBay в `/data/raw/ebay`. Решений нет.

## Результаты обучения

После **Права и эксплуатация** вы должны объяснить внутренний механизм, предсказать изменения metadata/files, выбрать безопасную команду и доказать итог измерением, а не сообщением об успехе.

## Архитектурная модель

Доступ требует execute на каждом родительском каталоге и permission/ACL на цели. Health включает capacity, live DataNodes, replication, corrupt blocks, quotas и recoverability.

```text
client ── metadata RPC ──► NameNode
  │                         │ block locations
  └── data stream ──► DataNode 1 ──► DataNode 2

HiveServer2 ──► Metastore (schema/location/partitions)
      └──────► execution engine ──► HDFS files
```
NameNode не хранит содержимое файла, а Metastore не хранит строки таблицы.

## Физическая схема eBay

```text
/data/raw/ebay/
├── snapshot_dt=2026-06-24/part-....snappy.parquet
├── snapshot_dt=2026-06-25/part-....snappy.parquet
└── ...
```
Grain: наблюдение `itemid` в `snapshot_dt`. Группы колонок: карточка/цена,
иерархия категорий, продавец, география и доставка. Полная schema — в `data-catalog`.

Общий raw read-only; результаты принадлежат `/user/$HDFS_USER/hadoop_training` и личной Hive DB.

## Алгоритм исследования

1. Зафиксируйте path/URI, owner и ожидаемый объект. 2. Снимите состояние до. 3. Выполните одно изменение. 4. Проверьте exit code. 5. Измерьте namespace/files/bytes/schema/rows. 6. Повторите команду и оцените идемпотентность. 7. Сохраните evidence.

Фиксируйте observed state, threshold, diagnosis, safe action и повторную проверку; chmod 777 запрещён как ответ.

## Типичные ошибки

- Путать локальный путь с HDFS URI.
- Делать вывод по `ls`, не проверяя blocks/bytes/schema.
- Использовать root или 777 вместо модели доступа.
- Создавать partition-каталог без Metastore или metadata без файлов.
- Считать replication резервной копией.
- Игнорировать малые файлы и цену NameNode metadata.

## Самопроверка

1. Какие metadata изменятся? 2. Где физически лежат bytes? 3. Сколько logical и physical bytes? 4. Кто может читать/писать? 5. Что произойдёт при повторе? 6. Какая независимая команда опровергнет вывод?

## Ментальная модель

HDFS проверяет owner/group/mode и ACL на каждом компоненте пути. Эксплуатация включает fsck, replication, quotas, capacity и восстановление; chmod 777 не является архитектурой доступа.

## Подробная теория

### 1. Авторизация

Для доступа нужны execute на всех родительских каталогах и нужное право на объекте. ACL дополняет POSIX mode, а mask ограничивает эффективные права.

### 2. Владение

chown — административная операция; приложениям обычно дают групповую роль или ACL. Общий raw должен быть read-only.

### 3. Quotas

Namespace quota ограничивает число имён, space quota — физический расход с учётом replication. Они останавливают runaway pipeline до заполнения диска.

### 4. Наблюдаемость

dfsadmin -report показывает узлы и capacity, fsck — блоки, JMX — метрики NameNode. Один зелёный endpoint не доказывает здоровье данных.

### 5. Восстановление

Trash, snapshots и replication решают разные задачи. Runbook должен содержать точные признаки сбоя, безопасные команды и проверку после восстановления.

## Стенд

NameNode `namenode:8020`, два DataNode, HiveServer2 `hiveserver2:10000`. Личные артефакты не создаются в общем read-only raw-слое.

## Как сдаётся задание

Валидатор проверяет артефакт и JSON-доказательство. В `command` запишите фактическую команду, в `observation` — измеренный результат, в `explanation` — почему он получился. Минимальная длина защищает от пустых ответов; содержательный смысл остаётся вашей ответственностью.

In [ ]:
import json, os, subprocess, tempfile

def save_evidence(module, task, command, observation, explanation):
    user=os.environ.get("HDFS_USER", os.environ.get("HADOOP_USER_NAME", "student"))
    target=f"/user/{user}/hadoop_training/evidence/{module}/task_{task:02d}.json"
    payload={"module":module,"task":task,"command":command,"observation":observation,"explanation":explanation}
    with tempfile.NamedTemporaryFile("w",encoding="utf-8",delete=False,suffix=".json") as f:
        json.dump(payload,f,ensure_ascii=False,indent=2); local=f.name
    subprocess.run(["hdfs","dfs","-mkdir","-p",target.rsplit("/",1)[0]],check=True)
    subprocess.run(["hdfs","dfs","-put","-f",local,target],check=True)
    os.unlink(local)
    print(target)

### Задание 1. POSIX mode

Создайте `/user/$HDFS_USER/hadoop_training/operations/task_01` с минимум одним файлом. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `POSIX mode`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py operations 1

### Задание 2. chmod symbolic

Создайте `/user/$HDFS_USER/hadoop_training/operations/task_02` с минимум одним файлом. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `chmod symbolic`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py operations 2

### Задание 3. chmod octal

Создайте `/user/$HDFS_USER/hadoop_training/operations/task_03` с минимум одним файлом. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `chmod octal`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py operations 3

### Задание 4. chown

Создайте `/user/$HDFS_USER/hadoop_training/operations/task_04` с минимум одним файлом. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `chown`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py operations 4

### Задание 5. chgrp

Создайте `/user/$HDFS_USER/hadoop_training/operations/task_05` с минимум одним файлом. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `chgrp`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py operations 5

### Задание 6. umask

Создайте `/user/$HDFS_USER/hadoop_training/operations/task_06` с минимум одним файлом. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `umask`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py operations 6

### Задание 7. ACL user

Создайте `/user/$HDFS_USER/hadoop_training/operations/task_07` с минимум одним файлом. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `ACL user`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py operations 7

### Задание 8. ACL group

Создайте `/user/$HDFS_USER/hadoop_training/operations/task_08` с минимум одним файлом. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `ACL group`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py operations 8

### Задание 9. default ACL

Создайте `/user/$HDFS_USER/hadoop_training/operations/task_09` с минимум одним файлом. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `default ACL`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py operations 9

### Задание 10. mask ACL

Создайте `/user/$HDFS_USER/hadoop_training/operations/task_10` с минимум одним файлом. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `mask ACL`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py operations 10

### Задание 11. getfacl

Создайте `/user/$HDFS_USER/hadoop_training/operations/task_11` с минимум одним файлом. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `getfacl`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py operations 11

### Задание 12. remove ACL

Создайте `/user/$HDFS_USER/hadoop_training/operations/task_12` с минимум одним файлом. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `remove ACL`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py operations 12

### Задание 13. shared raw layer

Создайте `/user/$HDFS_USER/hadoop_training/operations/task_13` с минимум одним файлом. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `shared raw layer`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py operations 13

### Задание 14. private user layer

Создайте `/user/$HDFS_USER/hadoop_training/operations/task_14` с минимум одним файлом. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `private user layer`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py operations 14

### Задание 15. sticky bit

Создайте `/user/$HDFS_USER/hadoop_training/operations/task_15` с минимум одним файлом. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `sticky bit`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py operations 15

### Задание 16. trash policy

Создайте `/user/$HDFS_USER/hadoop_training/operations/task_16` с минимум одним файлом. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `trash policy`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py operations 16

### Задание 17. space quota

Создайте `/user/$HDFS_USER/hadoop_training/operations/task_17` с минимум одним файлом. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `space quota`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py operations 17

### Задание 18. namespace quota

Создайте `/user/$HDFS_USER/hadoop_training/operations/task_18` с минимум одним файлом. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `namespace quota`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py operations 18

### Задание 19. setrep operations

Создайте `/user/$HDFS_USER/hadoop_training/operations/task_19` с минимум одним файлом. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `setrep operations`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py operations 19

### Задание 20. decommission concept

Создайте `/user/$HDFS_USER/hadoop_training/operations/task_20` с минимум одним файлом. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `decommission concept`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py operations 20

### Задание 21. balancer concept

Создайте `/user/$HDFS_USER/hadoop_training/operations/task_21` с минимум одним файлом. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `balancer concept`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py operations 21

### Задание 22. safemode check

Создайте `/user/$HDFS_USER/hadoop_training/operations/task_22` с минимум одним файлом. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `safemode check`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py operations 22

### Задание 23. fsck health

Создайте `/user/$HDFS_USER/hadoop_training/operations/task_23` с минимум одним файлом. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `fsck health`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py operations 23

### Задание 24. block locations

Создайте `/user/$HDFS_USER/hadoop_training/operations/task_24` с минимум одним файлом. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `block locations`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py operations 24

### Задание 25. DataNode report

Создайте `/user/$HDFS_USER/hadoop_training/operations/task_25` с минимум одним файлом. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `DataNode report`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py operations 25

### Задание 26. NameNode JMX

Создайте `/user/$HDFS_USER/hadoop_training/operations/task_26` с минимум одним файлом. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `NameNode JMX`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py operations 26

### Задание 27. audit logs

Создайте `/user/$HDFS_USER/hadoop_training/operations/task_27` с минимум одним файлом. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `audit logs`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py operations 27

### Задание 28. least privilege

Создайте `/user/$HDFS_USER/hadoop_training/operations/task_28` с минимум одним файлом. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `least privilege`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py operations 28

### Задание 29. recovery drill

Создайте `/user/$HDFS_USER/hadoop_training/operations/task_29` с минимум одним файлом. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `recovery drill`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py operations 29

### Задание 30. operations checklist

Создайте `/user/$HDFS_USER/hadoop_training/operations/task_30` с минимум одним файлом. **До запуска:** запишите ожидаемый результат. **После:** измерьте число файлов/строк, свойства или права, относящиеся к теме `operations checklist`. Проверьте повторный запуск и не изменяйте `/data/raw/ebay`.

<details><summary>Подсказка</summary>

Выполняйте одно изменение за шаг. Сравните состояние до/после независимой командой, затем сохраните evidence.

</details>

<!-- task-card-hadoop-v1 -->
#### Карточка выполнения

- **Цель:** объясните HDFS/Hive/Parquet-механизм задания.
- **Вход:** укажите URI, формат, owner, grain и объём.
- **Выход:** точный path/table, lifecycle и ожидаемые files/rows.
- **До/после:** снимите `test/stat/count/du/getfacl`, schema или EXPLAIN по теме.
- **Безопасность:** работайте только в личном каталоге; raw не изменяется.
- **Повтор:** сформулируйте, почему второй запуск безопасен или чем он отличается.
- **Evidence/checker:** сохраните фактическую команду, наблюдение и причинное объяснение.

Созданный пустой каталог и текст «команда сработала» не доказывают выполнение.

In [ ]:
# Ваши команды здесь
# Для shell используйте !hdfs dfs ... или %%bash
# Для Hive используйте !beeline ...
# save_evidence(...)

In [ ]:
!python3 /opt/lab/hadoop-training/check_task.py operations 30

## Итог

Все 30 проверок должны возвращать PASS. Удалять чужие или raw-данные запрещено.